# Compile a Test Set for a Processed Dataset

In [2]:
import sys
sys.path.append('../utils')
from json_parser import create_overlaid_img

import numpy as np
import os
import cv2
import pandas as pd
import shutil
import random
from typing import List, Dict, Union, Tuple

In [3]:
# the ouput folders for test images and annotations
# they will be created under the processed dataset folder specified below
# the images and annotations will share the same name
TEST_ANNOTATIONS_FOLDER = 'test_annotations'
TEST_IMAGES_FOLDER = 'test_images'

## Dataset configurations
The location of the processed dataset folder (as produced by the ML-OPs pipeline), the location of json annotation files, and also the location of images. It is possible to have one annotation applicable to multiple images. In this case, multiple folders with the images can be specified. Then, only one image is randomly selected and paired with the annotation file to be used for testing and the rest of images for that annotations will be ignored.

In [6]:
# The location of the processed dataset
# PROCESSED_DATA_PATH = "/home/cellareye/Cellanome/Data/20250227_preadipocytes-adhered_10x_caged"
PROCESSED_DATA_PATH = "/global/home/ashish.sinha/cellanome/datasets/20240221_jurkat_4x_caged/"
# The location of annotations
ANNOTATIONS_FOLDER = "annotations"
# The location of images to be considered; one annotation can be applied to multiple images
# in that case, an image is randomly selected and paired with the annotation to be used for testing
# (the rest of images for that annotations will be ignored)
IMAGES_FOLDERS = ["White_dz-12", "White_dz-9", "White_dz-6", "White_dz-3", "White_dz0", 
                  "White_dz3", "White_dz6", "White_dz9", "White_dz12"]
# IMAGES_FOLDERS = ["White_dz-32", "White_dz-24", "White_dz-16", "White_dz-8", "White_dz0", 
#                   "White_dz8", "White_dz16", "White_dz24", "White_dz32"]

FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY = None
Z_OFFSET_IDENTIFIER_TO_COLOR_FOR_OVERLAY = None

# IMAGES_FOLDERS = ["White_dz0"]
# FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY = {'Violet': 'red', 'Blue': 'blue'}
# Z_OFFSET_IDENTIFIER_TO_COLOR_FOR_OVERLAY = {'White_dz-6': 'red', 'White_dz0': 'green', 'White_dz6': 'blue'}

### Checks
Make sure the required files are included in the folder.

In [5]:
if len(IMAGES_FOLDERS) != 1 and 'annotation_images_mapping.csv' not in os.listdir(PROCESSED_DATA_PATH):
    print("[ERROR]: 'annotation_images_mapping.csv' file is needed in case one annotation file is applicable to many images")
    print("Fix the data before proceeding ...") 

FileNotFoundError: [Errno 2] No such file or directory: '/home/cellareye/Cellanome/Data/20250227_preadipocytes-adhered_10x_caged'

In [5]:
if Z_OFFSET_IDENTIFIER_TO_COLOR_FOR_OVERLAY is not None and len(IMAGES_FOLDERS) > 1:
    print("[ERROR]: Z-stack BF overlaid image can only be generated with IMAGES_FOLDERS containing the best focus image folder!")
    print("Fix the data before proceeding ...") 

In [6]:
if os.path.exists(os.path.join(PROCESSED_DATA_PATH, 'test.txt')):
    with open(os.path.join(PROCESSED_DATA_PATH, 'test.txt')) as file:
        test_annotation_files = file.readlines()
        test_annotation_files = [f.replace('\n', '') + '.json' for f in test_annotation_files if len(f) > 0]
else:
    print("[WARN]: No test.txt file could be found under the dataset folder. All the json annotations will be used for testing!")
    test_annotation_files = os.listdir(os.path.join(PROCESSED_DATA_PATH, ANNOTATIONS_FOLDER))

In [7]:
if not os.path.exists(os.path.join(PROCESSED_DATA_PATH, TEST_ANNOTATIONS_FOLDER)):
    os.mkdir(os.path.join(PROCESSED_DATA_PATH, TEST_ANNOTATIONS_FOLDER))
    print(f"[INFO]: Test annotation folder '{TEST_ANNOTATIONS_FOLDER}' created under the dataset folder")

else:
    print(f"[WARN]: The test annotation folder '{TEST_ANNOTATIONS_FOLDER}' already exists under the dataset folder! Delete the content")

if not os.path.exists(os.path.join(PROCESSED_DATA_PATH, TEST_IMAGES_FOLDER)):
    os.mkdir(os.path.join(PROCESSED_DATA_PATH, TEST_IMAGES_FOLDER))
    print(f"[INFO]: Test image folder '{TEST_IMAGES_FOLDER}' created under the dataset folder")
else:
    print(f"[WARN]: The image annotation folder '{TEST_IMAGES_FOLDER}' already exists under the dataset folder! Delete the content.")

[INFO]: Test annotation folder '_test_annotations' created under the dataset folder
[INFO]: Test image folder '_test_images' created under the dataset folder


In [8]:
if FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY is not None:
    OVERLAID_TEST_IMAGES_FOLDER = TEST_IMAGES_FOLDER + '_fl_overlaid'
    print(f"[INFO]: Generating FL overlaid test images ...")
    if not os.path.exists(os.path.join(PROCESSED_DATA_PATH, OVERLAID_TEST_IMAGES_FOLDER)):
        os.mkdir(os.path.join(PROCESSED_DATA_PATH, OVERLAID_TEST_IMAGES_FOLDER))
        print(f"[INFO]: FL overlaid image folder '{OVERLAID_TEST_IMAGES_FOLDER}' created under the dataset folder")
    else:
        print(f"[WARN]: FL overlaid image folder '{OVERLAID_TEST_IMAGES_FOLDER}' already exists under the dataset folder! Delete the content.")

if Z_OFFSET_IDENTIFIER_TO_COLOR_FOR_OVERLAY is not None:
    Z_STACK_OVERLAID_TEST_IMAGES_FOLDER = TEST_IMAGES_FOLDER + '_z_stack_overlaid'
    print(f"[INFO]: Generating z-stack overlaid test images ...")
    if not os.path.exists(os.path.join(PROCESSED_DATA_PATH, Z_STACK_OVERLAID_TEST_IMAGES_FOLDER)):
        os.mkdir(os.path.join(PROCESSED_DATA_PATH, Z_STACK_OVERLAID_TEST_IMAGES_FOLDER))
        print(f"[INFO]: Z-stack overlaid image folder '{Z_STACK_OVERLAID_TEST_IMAGES_FOLDER}' created under the dataset folder")
    else:
        print(f"[WARN]: Z-stack overlaid image folder '{Z_STACK_OVERLAID_TEST_IMAGES_FOLDER}' already exists under the dataset folder! Delete the content.")

[INFO]: Generating FL overlaid test images ...
[INFO]: FL overlaid image folder '_test_images_fl_overlaid' created under the dataset folder
[INFO]: Generating z-stack overlaid test images ...
[INFO]: Z-stack overlaid image folder '_test_images_z_stack_overlaid' created under the dataset folder


In [9]:
if os.path.exists(os.path.join(PROCESSED_DATA_PATH, 'annotation_images_mapping.csv')):
    annotations_images_map = pd.read_csv(os.path.join(PROCESSED_DATA_PATH, 'annotation_images_mapping.csv'))
    # remove any path in the json annotation files from 'annotation_images_mapping.csv' (no path is expected, but just in case)
    annotations_images_map['annotation_json'] = annotations_images_map['annotation_json'].apply(lambda x: os.path.basename(x))
    filtered_map = annotations_images_map[annotations_images_map['annotation_json'].apply(lambda x: True if x in test_annotation_files 
                                                                                          else False)]
    missing_files = [file for file in test_annotation_files if file not in filtered_map['annotation_json'].values]
    if len(missing_files) > 0:
        print("[ERROR]: The followine annotations specified in test.txt do not exist in the 'annotation_images_mapping.csv':")
        print("{missing_files}")
    
    if len(IMAGES_FOLDERS) == 1:
        # use the specified image folder for test (no random assignment)
        if IMAGES_FOLDERS[0] in annotations_images_map.columns:
            for _, row in filtered_map.iterrows():
                annotation_file = row['annotation_json']
                filename = ".".join(annotation_file.strip().split('.')[:-1])
                image_with_path = row[IMAGES_FOLDERS[0]]
                
                # use the same name as the image name for the annotation (with .jpg extension)
                image_file = os.path.basename(image_with_path)
                filename = ".".join(image_file.strip().split('.')[:-1])
                shutil.copy(os.path.join(PROCESSED_DATA_PATH, ANNOTATIONS_FOLDER, annotation_file), 
                            os.path.join(PROCESSED_DATA_PATH, TEST_ANNOTATIONS_FOLDER, filename + '.json'))
                shutil.copy(os.path.join(PROCESSED_DATA_PATH, image_with_path), 
                            os.path.join(PROCESSED_DATA_PATH, TEST_IMAGES_FOLDER, image_file))

                if FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY is not None:
                    # create BF and FL channels overlaid images for testing as well
                    bf_image: np.ndarray = cv2.imread(os.path.join(PROCESSED_DATA_PATH, image_with_path), cv2.IMREAD_UNCHANGED) 
                    
                    fl_images_dict = {}
                    for k, v in FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY.items():
                        if k in filtered_map.columns:
                            fl_image_with_path: str = row[k]
                            fl_images_dict[v]: np.ndarray = cv2.imread(os.path.join(PROCESSED_DATA_PATH, fl_image_with_path), cv2.IMREAD_UNCHANGED) 
                        else:
                            print(f"[WARN]: FL channel name {k} could not be found in 'annotation_images_mapping.csv'! Ignoring the channel for generating the overlaid image.")
                            
                    overlaid_img: np.ndarray = create_overlaid_img(bf_image=bf_image, fl_images_dict=fl_images_dict, normalize_fl_image_hist=True)
                    # we use the same image name (to be consistent with the annotations, this is the BF image name) but save at a different folder
                    cv2.imwrite(os.path.join(PROCESSED_DATA_PATH, OVERLAID_TEST_IMAGES_FOLDER, image_file), cv2.cvtColor(overlaid_img, cv2.COLOR_BGR2RGB))
                
                if Z_OFFSET_IDENTIFIER_TO_COLOR_FOR_OVERLAY is not None:
                    overlay_images_dict = {}
                    for k, v in Z_OFFSET_IDENTIFIER_TO_COLOR_FOR_OVERLAY.items():
                        if k in filtered_map.columns:
                            z_stack_image_with_path: str = row[k]
                            overlay_images_dict[v]: np.ndarray = cv2.imread(os.path.join(PROCESSED_DATA_PATH, z_stack_image_with_path), cv2.IMREAD_UNCHANGED) 
                        else:
                            print(f"[WARN]: Z-stack image name {k} could not be found in 'annotation_images_mapping.csv'! Ignoring the z-stack for generating the overlaid image.")
                    # we should not apply histogram normalization when overlaying z-stack images        
                    overlaid_img: np.ndarray = create_overlaid_img(bf_image=None, fl_images_dict=overlay_images_dict, normalize_fl_image_hist=False)
                    # we use the same image name (to be consistent with the annotations, this is the BF image name) but save at a different folder
                    cv2.imwrite(os.path.join(PROCESSED_DATA_PATH, Z_STACK_OVERLAID_TEST_IMAGES_FOLDER, image_file), cv2.cvtColor(overlaid_img, cv2.COLOR_BGR2RGB))

                    
        else:
            print(f"[ERROR] The specified image folder:  {IMAGES_FOLDERS[0]} does not exist in 'annotation_images_mapping.csv'!")
            print("Fix the IMAGES_FOLDERS and start again")
    else:
        # randomly select one image
        valid_image_folders = [folder for folder in IMAGES_FOLDERS if folder in annotations_images_map.columns]
        
        if 0 < len(valid_image_folders) < len(IMAGES_FOLDERS):
            print(f"[WARN] The following folders specfied in IMAGE_FOLDERS do not exist in 'annotation_images_mapping.csv':",  
                  f"{[folder for folder in IMAGES_FOLDERS if folder not in annotations_images_map.columns]}! Continue with the rest ...")
        
        if len(valid_image_folders) > 0:
            for _, row in filtered_map.iterrows():
                annotation_file = row['annotation_json']
                image_with_path = row[random.choice(valid_image_folders)]
                
                # use the same name as the image name for the annotation (with .jpg extension)
                image_file = os.path.basename(image_with_path)
                filename = ".".join(image_file.strip().split('.')[:-1])
                shutil.copy(os.path.join(PROCESSED_DATA_PATH, ANNOTATIONS_FOLDER, annotation_file), 
                            os.path.join(PROCESSED_DATA_PATH, TEST_ANNOTATIONS_FOLDER, filename + '.json'))
                shutil.copy(os.path.join(PROCESSED_DATA_PATH, image_with_path), 
                            os.path.join(PROCESSED_DATA_PATH, TEST_IMAGES_FOLDER, image_file))
                
                if FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY is not None:
                    # create BF and FL channels overlaid images for testing as well
                    
                    bf_image: np.ndarray = cv2.imread(os.path.join(PROCESSED_DATA_PATH, image_with_path), cv2.IMREAD_UNCHANGED) 
                    
                    fl_images_dict = {}
                    for k, v in FL_CH_IDENTIFIER_TO_COLOR_FOR_OVERLAY.items():
                        if k in filtered_map.columns:
                            fl_image_with_path: str = row[k]
                            fl_images_dict[v]: np.ndarray = cv2.imread(os.path.join(PROCESSED_DATA_PATH, fl_image_with_path), cv2.IMREAD_UNCHANGED)
                        else:
                            print(f"[WARN]: FL channel name {k} could not be found in 'annotation_images_mapping.csv'! Ignoring the channel for generating the overlaid image.")

                    overlaid_img: np.ndarray = create_overlaid_img(bf_image=bf_image, fl_images_dict=fl_images_dict, normalize_fl_image_hist=True)
                    # we use the same image name (to be consistent with the annotations, this is the BF image name) but save at a different folder
                    cv2.imwrite(os.path.join(PROCESSED_DATA_PATH, OVERLAID_TEST_IMAGES_FOLDER, image_file), cv2.cvtColor(overlaid_img, cv2.COLOR_BGR2RGB))
            
        else:
            print(f"[ERROR] None of the specified image folders:  {IMAGES_FOLDERS} exist in 'annotation_images_mapping.csv'!")
            print("Fix the IMAGES_FOLDERS and start again")

else:
    print("[INFO]:'annotation_images_mapping.csv' does not exist in the dataset folder!")
    # without the 'annotation_images_mapping.csv', the assumption is the image and the annotation share the same name
    # (without the extension)
    # also, len(IMAGES_FOLDERS) should be one
    image_files = os.listdir(os.path.join(PROCESSED_DATA_PATH, IMAGES_FOLDERS[0]))
    for annotation_file in test_annotation_files:
        filename = ".".join(annotation_file.strip().split('.')[:-1])
        if filename + '.jpg' in image_files:
            shutil.copy(os.path.join(PROCESSED_DATA_PATH, ANNOTATIONS_FOLDER, annotation_file), 
                        os.path.join(PROCESSED_DATA_PATH, TEST_ANNOTATIONS_FOLDER, annotation_file))
            shutil.copy(os.path.join(PROCESSED_DATA_PATH, IMAGES_FOLDERS[0], filename + '.jpg'), 
                        os.path.join(PROCESSED_DATA_PATH, TEST_IMAGES_FOLDER, filename + '.jpg'))
        else:
            print(f"[WARN]: No image with name {filename}.jpg could be found for annotation file {annotation_file} in ", 
                  f"{IMAGES_FOLDERS[0]} folder! Skipping the annotation ... ")